# Bài tập 2 - Mạng Neural Đơn giản với TensorFlow
Trong bài thực hành này, chúng ta sẽ xây dựng một mạng neural nhỏ sử dụng TensorFlow.
   <center> <img  src="./images/C2_W1_CoffeeRoasting.png" width="400" />   <center/>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from lab_utils_common import dlc
from lab_coffee_utils import load_coffee_data, plt_roast, plt_prob, plt_layer, plt_network, plt_output_unit
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)


## Bộ dữ liệu (DataSet)

In [ ]:
X,Y = load_coffee_data();
print(X.shape, Y.shape)

Hãy vẽ biểu đồ dữ liệu rang cà phê bên dưới. Hai đặc trưng là Nhiệt độ theo độ C và Thời lượng theo phút. Trang [Coffee Roasting at Home](https://www.merchantsofgreencoffee.com/how-to-roast-green-coffee-in-your-oven/) gợi ý rằng thời lượng tốt nhất nên duy trì trong khoảng 12 đến 15 phút trong khi nhiệt độ nên nằm trong khoảng 175 đến 260 độ C. Tất nhiên, khi nhiệt độ tăng lên thì thời lượng nên giảm đi. 

In [ ]:
plt_roast(X,Y)

### Chuẩn hóa Dữ liệu (Normalize Data)
Việc khớp các trọng số với dữ liệu (lan truyền ngược - back-propagation) sẽ diễn ra nhanh hơn nếu dữ liệu được chuẩn hóa. Đây cũng là quy trình tương tự đối với các mô hình hồi quy, nơi các đặc trưng trong dữ liệu đều được chuẩn hóa để có phạm vi giá trị tương đương nhau. 
Quy trình bên dưới sử dụng một [lớp chuẩn hóa](https://keras.io/api/layers/preprocessing_layers/numerical/normalization/) của Keras. Nó bao gồm các bước sau:
- Tạo một "Lớp Chuẩn Hóa" (Normalization Layer). Lưu ý, trong cách áp dụng ở đây, đây không phải là một lớp trong mô hình của bạn.
- 'adapt' dữ liệu. Bước này sẽ học giá trị trung bình (mean) và phương sai (variance) của tập dữ liệu rồi lưu lại các giá trị đó vào bên trong.
- Chuẩn hóa dữ liệu.
Điều quan trọng là phải áp dụng chuẩn hóa cho bất kỳ dữ liệu tương lai nào sử dụng mô hình đã học.

In [ ]:
print(f"Temperature Max, Min pre normalization: {np.max(X[:,0]):0.2f}, {np.min(X[:,0]):0.2f}")
print(f"Duration    Max, Min pre normalization: {np.max(X[:,1]):0.2f}, {np.min(X[:,1]):0.2f}")
norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X)  # learns mean, variance
Xn = norm_l(X)
print(f"Temperature Max, Min post normalization: {np.max(Xn[:,0]):0.2f}, {np.min(Xn[:,0]):0.2f}")
print(f"Duration    Max, Min post normalization: {np.max(Xn[:,1]):0.2f}, {np.min(Xn[:,1]):0.2f}")

Sao chép/nhân bản (Tile/copy) dữ liệu của chúng ta để tăng kích thước tập huấn luyện và giảm số lượng epoch cần huấn luyện.

In [ ]:
Xt = np.tile(Xn,(1000,1))
Yt= np.tile(Y,(1000,1))   
print(Xt.shape, Yt.shape)   

## Mô hình TensorFlow

### Mô hình
   <center> <img  src="./images/C2_W1_RoastingNetwork.PNG" width="200" />   <center/>  
Hãy cùng xây dựng mạng "Coffee Roasting Network". Có hai lớp với hàm kích hoạt sigmoid như minh họa dưới đây:

In [ ]:
tf.random.set_seed(1234)  # applied to achieve consistent results
model = Sequential(
    [
        tf.keras.Input(shape=(2,)),
        Dense(3, activation='sigmoid', name = 'layer1'),
        Dense(1, activation='sigmoid', name = 'layer2')
     ]
)

>**Lưu ý 1:** Dòng lệnh `tf.keras.Input(shape=(2,)),` chỉ định kích thước dự kiến của đầu vào. Điều này cho phép TensorFlow xác định kích thước của các tham số trọng số và bias ngay tại thời điểm này. Điều này rất hữu ích khi tìm hiểu các mô hình TensorFlow. Câu lệnh này có thể được lược bỏ trong thực tế và TensorFlow sẽ tự định kích thước các tham số mạng khi dữ liệu đầu vào được chỉ định trong câu lệnh `model.fit`.

>**Lưu ý 2:** Việc đưa hàm kích hoạt sigmoid trực tiếp vào lớp cuối cùng không được xem là phương pháp tối ưu nhất (best practice). Thay vào đó, nó nên được tính gộp vào trong hàm mất mát để cải thiện độ ổn định về mặt số học.

Hàm `model.summary()` cung cấp phần mô tả chi tiết về mạng:

In [ ]:
model.summary()

Số lượng tham số hiển thị trong phần tóm tắt (summary) tương ứng với số phần tử trong các mảng trọng số và bias như minh họa dưới đây.

In [ ]:
L1_num_params = 2 * 3 + 3   # W1 parameters  + b1 parameters
L2_num_params = 3 * 1 + 1   # W2 parameters  + b2 parameters
print("L1 params = ", L1_num_params, ", L2 params = ", L2_num_params  )

Hãy cùng kiểm tra các trọng số và bias mà TensorFlow đã khởi tạo. Trọng số $W$ phải có kích thước là (số đặc trưng ở đầu vào, số đơn vị trong lớp) trong khi kích thước của bias $b$ phải khớp với số đơn vị trong lớp:
- Ở lớp đầu tiên với 3 đơn vị, chúng ta kỳ vọng W có kích thước là (2,3) và $b$ có 3 phần tử.
- Ở lớp thứ hai với 1 đơn vị, chúng ta kỳ vọng W có kích thước là (3,1) và $b$ có 1 phần tử.

In [ ]:
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print(f"W1{W1.shape}:\n", W1, f"\nb1{b1.shape}:", b1)
print(f"W2{W2.shape}:\n", W2, f"\nb2{b2.shape}:", b2)

Các câu lệnh sau đây sẽ được mô tả chi tiết trong Tuần 2. Hiện tại:
- Câu lệnh `model.compile` định nghĩa một hàm mất mát và chỉ định thuật toán tối ưu hóa khi biên dịch.
- Câu lệnh `model.fit` thực thi thuật toán hạ gradient và khớp các trọng số với dữ liệu.

In [ ]:
model.compile(
    loss = tf.keras.losses.BinaryCrossentropy(),
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01),
)

model.fit(
    Xt,Yt,            
    epochs=10,
)

#### Epochs và batches
Trong câu lệnh `fit` ở trên, số lượng `epochs` được đặt là 10. Điều này có nghĩa rằng toàn bộ tập dữ liệu sẽ được duyệt qua trong quá trình huấn luyện 10 lần. Trong quá trình huấn luyện, bạn thấy đầu ra mô tả tiến trình huấn luyện trông như sau:
```
Epoch 1/10
6250/6250 [==============================] - 6s 910us/step - loss: 0.1782
```
Dòng đầu tiên, `Epoch 1/10`, mô tả mô hình hiện đang chạy ở epoch nào. Để đạt hiệu quả cao, tập dữ liệu huấn luyện được chia thành các 'batches' (lô dữ liệu). Kích thước mặc định của một batch trong TensorFlow là 32. Có 200.000 mẫu trong tập dữ liệu mở rộng của chúng ta, tương đương 6.250 batches. Ký hiệu ở dòng thứ hai `6250/6250 [====` mô tả batch nào đã được thực thi.

#### Các trọng số đã được cập nhật
Sau khi khớp mô hình (fitting), các trọng số đã được cập nhật: 

In [ ]:
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print("W1:\n", W1, "\nb1:", b1)
print("W2:\n", W2, "\nb2:", b2)

Bạn có thể thấy rằng các giá trị đã khác so với những gì bạn đã in ra trước khi gọi `model.fit()`. Với những trọng số này, mô hình sẽ có thể phân biệt được mẻ cà phê nào là rang đạt chuẩn hay không đạt chuẩn.

Để phục vụ cho phần thảo luận tiếp theo, thay vì sử dụng ngay các trọng số bạn vừa nhận được, trước tiên bạn sẽ thiết lập một số trọng số mà chúng ta đã lưu từ một lần huấn luyện trước đó. Điều này nhằm giúp notebook này hoạt động ổn định và nhất quán trước các thay đổi của TensorFlow theo thời gian. Các lần huấn luyện khác nhau có thể tạo ra các kết quả hơi khác nhau một chút và phần thảo luận sau đây áp dụng khi mô hình có các trọng số mà bạn sẽ tải dưới đây. 

Sau này, bạn có thể tự do chạy lại notebook với ô bên dưới được chú thích (comment out) để xem có sự khác biệt nào không. Nếu bạn đạt được độ mất mát thấp sau quá trình huấn luyện ở trên (ví dụ: 0.002), thì khả năng rất cao là bạn sẽ nhận được cùng kết quả.

In [ ]:
# After finishing the lab later, you can re-run all 
# cells except this one to see if your trained model
# gets the same results.

# Set weights from a previous run. 
W1 = np.array([
    [-8.94,  0.29, 12.89],
    [-0.17, -7.34, 10.79]] )
b1 = np.array([-9.87, -9.28,  1.01])
W2 = np.array([
    [-31.38],
    [-27.86],
    [-32.79]])
b2 = np.array([15.54])

# Replace the weights from your trained model with
# the values above.
model.get_layer("layer1").set_weights([W1,b1])
model.get_layer("layer2").set_weights([W2,b2])

In [ ]:
# Check if the weights are successfully replaced
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print("W1:\n", W1, "\nb1:", b1)
print("W2:\n", W2, "\nb2:", b2)

### Dự đoán (Predictions)
<img align="left" src="./images/C2_W1_RoastingDecision.PNG"     style=" width:380px; padding: 10px 20px; " >

Khi đã có một mô hình được huấn luyện, bạn có thể sử dụng nó để đưa ra dự đoán. Hãy nhớ rằng đầu ra của mô hình chúng ta là một xác suất. Trong trường hợp này, đó là xác suất mẻ rang đạt chuẩn (good roast). Để đưa ra quyết định, ta phải so sánh xác suất với một ngưỡng (threshold). Ở đây, chúng ta sẽ sử dụng ngưỡng 0.5

Hãy bắt đầu bằng cách tạo dữ liệu đầu vào. Mô hình kỳ vọng nhận vào một hoặc nhiều mẫu thử, trong đó các mẫu nằm trên các hàng của ma trận. Trong trường hợp này, chúng ta có hai đặc trưng nên ma trận sẽ có kích thước (m,2) với m là số lượng mẫu.
Hãy nhớ rằng, chúng ta đã chuẩn hóa các đặc trưng đầu vào nên chúng ta cũng phải chuẩn hóa dữ liệu kiểm tra của mình.   
Để đưa ra dự đoán, bạn áp dụng phương thức `predict`.

In [ ]:
X_test = np.array([
    [200,13.9],  # positive example
    [200,17]])   # negative example
X_testn = norm_l(X_test)
predictions = model.predict(X_testn)
print("predictions = \n", predictions)

Để chuyển đổi các xác suất thành quyết định, chúng ta áp dụng một ngưỡng:

In [ ]:
yhat = np.zeros_like(predictions)
for i in range(len(predictions)):
    if predictions[i] >= 0.5:
        yhat[i] = 1
    else:
        yhat[i] = 0
print(f"decisions = \n{yhat}")

Điều này có thể được thực hiện một cách ngắn gọn hơn:

In [ ]:
yhat = (predictions >= 0.5).astype(int)
print(f"decisions = \n{yhat}")

## Các hàm của Lớp (Layer Functions)
Hãy cùng kiểm tra chức năng của các đơn vị (units) để xác định vai trò của chúng trong quyết định chất lượng rang cà phê. Chúng ta sẽ vẽ biểu đồ đầu ra của từng node đối với mọi giá trị của các đặc trưng đầu vào (thời lượng, nhiệt độ). Mỗi đơn vị là một hàm logistic có đầu ra dao động từ 0 đến 1. Vùng bóng mờ trong biểu đồ biểu diễn giá trị đầu ra.
> Lưu ý: Trong các bài thực hành, chúng ta thường đánh số bắt đầu từ 0.

In [ ]:
plt_layer(X,Y.reshape(-1,),W1,b1,norm_l)

Vùng bóng mờ cho thấy mỗi đơn vị chịu trách nhiệm cho một vùng "rang hỏng" (bad roast) khác nhau. Đơn vị 0 có giá trị lớn hơn khi nhiệt độ quá thấp. Đơn vị 1 có giá trị lớn hơn khi thời lượng quá ngắn và đơn vị 2 có giá trị lớn hơn đối với các kết hợp thời gian/nhiệt độ không tốt. Điều đáng chú ý là mạng đã tự học được các hàm này thông qua quá trình hạ gradient. Chúng rất giống với loại chức năng mà con người có thể lựa chọn để đưa ra các quyết định tương tự.

Biểu đồ hàm số của lớp cuối cùng hơi khó hình dung hơn một chút. Đầu vào của nó là đầu ra của lớp thứ nhất. Chúng ta biết rằng lớp thứ nhất sử dụng hàm sigmoid nên phạm vi đầu ra của chúng nằm trong khoảng từ 0 đến 1. Chúng ta có thể tạo biểu đồ 3D tính toán đầu ra cho mọi tổ hợp có thể có của ba đầu vào. Điều này được thể hiện bên dưới. Ở trên, các giá trị đầu ra cao tương ứng với các vùng 'rang hỏng'. Ở bên dưới, đầu ra cực đại nằm ở các vùng mà cả ba đầu vào đều có giá trị nhỏ, tương ứng với các vùng 'rang đạt chuẩn'.

In [ ]:
plt_output_unit(W2,b2)

Biểu đồ cuối cùng thể hiện toàn bộ mạng đang hoạt động.  
Biểu đồ bên trái là đầu ra thô của lớp cuối cùng, được biểu diễn bằng vùng màu xanh lam. Biểu đồ này được phủ lên trên dữ liệu huấn luyện (được biểu thị bằng các dấu X và O).   
Biểu đồ bên phải là đầu ra của mạng sau khi áp dụng ngưỡng quyết định. Các dấu X và O ở đây tương ứng với các quyết định do mạng đưa ra.  
Đoạn code sau đây sẽ mất một chút thời gian để chạy:

In [ ]:
netf= lambda x : model.predict(norm_l(x))
plt_network(X,Y,netf)

## Chúc mừng bạn!
Bạn đã xây dựng một mạng neural đơn giản bằng TensorFlow. 
Mạng này đã chứng minh khả năng của các mạng neural trong việc xử lý các quyết định phức tạp bằng cách phân chia quyết định giữa nhiều đơn vị.

*Nguồn: Andrew Ng, Advanced Learning Algorithms, DeepLearning.AI & Standford Online.*